# Comparaison des performances : modèle génératif (LLM) vs BERT

Ce notebook compare les performances de deux classifieurs sur la tâche de classification entre les étiquettes
**Éditorial** et **Publicité** :

- un **modèle génératif** (LLM) interrogé avec `prompt_classification_llm.txt` —
  export attendu : `results/generations.csv` ;
- un **modèle BERT** entraîné dans ActiveTigger — export attendu :
  `results/predictions_BERT.csv`.

## Point important : le filtrage sur le jeu de test

Les deux fichiers contiennent des prédictions sur **toutes** les lignes qui leur ont
été soumises (entraînement compris), pas seulement sur le jeu de test. On ne peut
évaluer un modèle que sur des lignes qu'il n'a **jamais vues pendant
l'entraînement** : ce notebook filtre donc chaque fichier pour ne garder que les
lignes marquées comme jeu de test (`batch` commençant par `test` pour le modèle
génératif, `dataset == "test"` pour BERT).

## Second point important : la vérité terrain

Pour calculer une métrique de performance (accuracy, F1...), il faut connaître le
**vrai label** de chaque ligne de test, pas seulement la prédiction. Ce notebook va
chercher cette vérité terrain dans `corpus_annotation_pre_annote_300.csv` (les 300
premières lignes annotées). **Si vous avez annoté davantage de lignes dans
ActiveTigger depuis, ou dans un autre fichier, la vérité de certaines lignes de test
ne sera pas trouvée** — le notebook le signale clairement et exclut ces lignes du
calcul plutôt que d'inventer un résultat. Ajustez `FICHIER_REFERENCE` dans la
cellule de paramètres si votre fichier de vérité terrain a un autre nom ou couvre
davantage de lignes.

## Comment ça marche : la clé de correspondance

Les trois fichiers (les deux exports de prédictions + le fichier de référence) sont
tous indexés sur les **mêmes 750 lignes de `corpus_annotation.csv`, dans le même
ordre**. La colonne `row_number` des exports correspond donc directement à la
position de la ligne dans ce fichier d'origine (et dans le fichier de référence).
C'est cette colonne qui sert à tout relier, plutôt que le texte brut (plus fragile
à cause des variations d'encodage entre outils).

## Comment utiliser ce notebook

1. Déposez `generations.csv` et `predictions_BERT.csv` dans le dossier `results/`.
2. Vérifiez la cellule **« Paramètres »** (noms de fichiers, de colonnes).
3. Exécutez les cellules dans l'ordre. Chaque étape affiche un résumé chiffré pour
   vérifier que le filtrage se passe comme attendu.


## 1. Installation et imports

In [ ]:
%pip install -q pandas scikit-learn matplotlib


In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
)
import matplotlib.pyplot as plt

# Couleurs fixes par modèle (palette sans danger pour le daltonisme), réutilisées
# dans tous les graphiques du notebook pour que chaque modèle garde toujours la
# même couleur.
COULEUR_GENERATIF = "#0072B2"   # bleu
COULEUR_BERT = "#E69F00"        # orange

print("Imports OK")


## 2. Paramètres

Noms de fichiers et de colonnes à adapter si vos exports diffèrent de ceux utilisés
ici.


In [ ]:
# --- Fichiers d'entrée (dans le dossier results/) ---
FICHIER_GENERATIF = "results/generations.csv"
FICHIER_BERT = "results/predictions_BERT.csv"

# --- Fichier contenant la vérité terrain (vrai label de chaque ligne) ---
FICHIER_REFERENCE = "corpus_annotation_pre_annote_300.csv"
COLONNE_LABEL_REFERENCE = "label"

# --- Colonnes des fichiers de prédiction ---
COLONNE_BATCH_GENERATIF = "batch"        # ex. "test__..." / "train__..."
COLONNE_PREDICTION_GENERATIF = "answer"
COLONNE_DATASET_BERT = "dataset"         # ex. "train" / "valid" / "test"
COLONNE_PREDICTION_BERT = "prediction"

# --- Noms affichés dans les graphiques et tableaux ---
NOM_GENERATIF = "Modèle génératif (LLM)"
NOM_BERT = "BERT (ActiveTigger)"

print("Paramètres chargés.")


## 3. Chargement de la vérité terrain

On charge le fichier de référence et on lui attribue un `row_number` égal à sa
position (ligne 0, 1, 2...), pour pouvoir le relier aux prédictions.


In [ ]:
df_reference = pd.read_csv(FICHIER_REFERENCE)
df_reference["row_number"] = df_reference.index

NB_LIGNES_ATTENDU = 750  # nombre de lignes de corpus_annotation.csv

if len(df_reference) != NB_LIGNES_ATTENDU:
    print(
        f"[AVERTISSEMENT] '{FICHIER_REFERENCE}' contient {len(df_reference)} lignes, "
        f"alors que corpus_annotation.csv en contient {NB_LIGNES_ATTENDU}. "
        "Le rapprochement se fait par position (row_number = numéro de ligne), donc "
        "si ce fichier ne contient pas les 750 lignes DANS LE MÊME ORDRE que "
        "corpus_annotation.csv (par ex. un export ActiveTigger qui ne garde que les "
        "lignes annotées), les vérités seront associées aux mauvaises lignes sans "
        "erreur visible. Vérifiez l'origine de ce fichier avant de faire confiance "
        "aux résultats ci-dessous."
    )

nb_avec_verite = df_reference[COLONNE_LABEL_REFERENCE].notna().sum()
print(f"Fichier de référence : {len(df_reference)} lignes au total.")
print(f"Lignes avec une vérité terrain connue : {nb_avec_verite}.")
print(f"Répartition des vérités connues :")
print(df_reference[COLONNE_LABEL_REFERENCE].value_counts())


## 4. Chargement et filtrage des prédictions du modèle génératif

On ne garde que les lignes dont le `batch` commence par `test`.


In [ ]:
df_gen_brut = pd.read_csv(FICHIER_GENERATIF)
print(f"'{FICHIER_GENERATIF}' : {len(df_gen_brut)} lignes au total.")
print("Répartition par batch :")
print(df_gen_brut[COLONNE_BATCH_GENERATIF].apply(lambda b: str(b).split('__')[0]).value_counts())

df_gen_test = df_gen_brut[
    df_gen_brut[COLONNE_BATCH_GENERATIF].astype(str).str.startswith("test")
].copy()
df_gen_test = df_gen_test.rename(columns={COLONNE_PREDICTION_GENERATIF: "prediction_generatif"})
df_gen_test = df_gen_test[["row_number", "prediction_generatif"]]

print(f"\nLignes de test conservées pour le modèle génératif : {len(df_gen_test)}.")


## 5. Chargement et filtrage des prédictions BERT

On ne garde que les lignes dont le `dataset` vaut `test`.


In [ ]:
df_bert_brut = pd.read_csv(FICHIER_BERT)
print(f"'{FICHIER_BERT}' : {len(df_bert_brut)} lignes au total.")
print("Répartition par dataset :")
print(df_bert_brut[COLONNE_DATASET_BERT].value_counts())

df_bert_test = df_bert_brut[df_bert_brut[COLONNE_DATASET_BERT] == "test"].copy()
df_bert_test = df_bert_test.rename(columns={COLONNE_PREDICTION_BERT: "prediction_bert"})
df_bert_test = df_bert_test[["row_number", "prediction_bert"]]

print(f"\nLignes de test conservées pour BERT : {len(df_bert_test)}.")


## 6. Reconstitution du jeu d'évaluation commun

On rassemble, pour chaque `row_number` du jeu de test, la vérité terrain et les
deux prédictions — puis on écarte les lignes dont la vérité terrain est inconnue
(absente de `corpus_annotation_pre_annote_300.csv`).


In [ ]:
df_verite = df_reference[["row_number", COLONNE_LABEL_REFERENCE]].rename(
    columns={COLONNE_LABEL_REFERENCE: "verite"}
)

df_eval = df_gen_test.merge(df_bert_test, on="row_number", how="outer")
df_eval = df_eval.merge(df_verite, on="row_number", how="left")

nb_test_total = len(df_eval)
df_eval_valide = df_eval[df_eval["verite"].notna()].copy()
nb_test_valide = len(df_eval_valide)

print(f"Lignes de test au total (génératif + BERT réunis) : {nb_test_total}")
print(f"Lignes de test avec une vérité terrain connue       : {nb_test_valide}")
print(f"Lignes exclues faute de vérité terrain              : {nb_test_total - nb_test_valide}")

if nb_test_valide == 0:
    raise ValueError(
        "Aucune ligne de test n'a de vérité terrain connue dans FICHIER_REFERENCE. "
        "Vérifiez que ce fichier couvre bien les lignes de test (même row_number), "
        "ou pointez FICHIER_REFERENCE vers un export plus complet."
    )

df_eval_valide.head()


## 7. Calcul des métriques de performance

Accuracy, precision/recall/F1 (par classe et en moyenne macro) et matrice de
confusion, pour chaque modèle, sur le même jeu de lignes évaluables.


In [ ]:
def calculer_metriques(y_vrai, y_pred, nom_modele):
    """Calcule et affiche les métriques de classification pour un modèle."""
    accuracy = accuracy_score(y_vrai, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_vrai, y_pred, average="macro", zero_division=0
    )

    print(f"=== {nom_modele} ===")
    print(f"Accuracy       : {accuracy:.3f}")
    print(f"Precision (macro) : {precision:.3f}")
    print(f"Recall (macro)    : {recall:.3f}")
    print(f"F1 (macro)        : {f1:.3f}")
    print()
    print(classification_report(y_vrai, y_pred, zero_division=0))

    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}


resultats_generatif = calculer_metriques(
    df_eval_valide["verite"], df_eval_valide["prediction_generatif"], NOM_GENERATIF
)
print()
resultats_bert = calculer_metriques(
    df_eval_valide["verite"], df_eval_valide["prediction_bert"], NOM_BERT
)


## 8. Comparaison visuelle des métriques

In [ ]:
metriques = ["accuracy", "precision", "recall", "f1"]
valeurs_generatif = [resultats_generatif[m] for m in metriques]
valeurs_bert = [resultats_bert[m] for m in metriques]

x = range(len(metriques))
largeur = 0.35

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([i - largeur / 2 for i in x], valeurs_generatif, largeur,
       label=NOM_GENERATIF, color=COULEUR_GENERATIF)
ax.bar([i + largeur / 2 for i in x], valeurs_bert, largeur,
       label=NOM_BERT, color=COULEUR_BERT)

ax.set_xticks(list(x))
ax.set_xticklabels(["Accuracy", "Precision\n(macro)", "Recall\n(macro)", "F1\n(macro)"])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title(f"Comparaison des performances (n={nb_test_valide} lignes de test évaluables)")
ax.legend()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for i, v in enumerate(valeurs_generatif):
    ax.text(i - largeur / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)
for i, v in enumerate(valeurs_bert):
    ax.text(i + largeur / 2, v + 0.02, f"{v:.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


## 9. Matrices de confusion

In [ ]:
etiquettes = sorted(df_eval_valide["verite"].unique())

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

for ax, y_pred, nom, cmap in [
    (axes[0], df_eval_valide["prediction_generatif"], NOM_GENERATIF, "Blues"),
    (axes[1], df_eval_valide["prediction_bert"], NOM_BERT, "Oranges"),
]:
    cm = confusion_matrix(df_eval_valide["verite"], y_pred, labels=etiquettes)
    im = ax.imshow(cm, cmap=cmap)
    ax.set_xticks(range(len(etiquettes)))
    ax.set_yticks(range(len(etiquettes)))
    ax.set_xticklabels(etiquettes, rotation=45, ha="right")
    ax.set_yticklabels(etiquettes)
    ax.set_xlabel("Prédiction")
    ax.set_ylabel("Vérité terrain")
    ax.set_title(nom)
    for i in range(len(etiquettes)):
        for j in range(len(etiquettes)):
            couleur_texte = "white" if cm[i, j] > cm.max() / 2 else "black"
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", color=couleur_texte)

plt.tight_layout()
plt.show()


## 10. Où les deux modèles ne sont-ils pas d'accord avec la vérité, ou entre eux ?

Utile pour discuter en formation des cas difficiles : erreurs communes aux deux
modèles, erreurs propres à un seul, désaccords entre les deux modèles.


In [ ]:
df_eval_valide["generatif_correct"] = df_eval_valide["prediction_generatif"] == df_eval_valide["verite"]
df_eval_valide["bert_correct"] = df_eval_valide["prediction_bert"] == df_eval_valide["verite"]
df_eval_valide["modeles_d_accord"] = df_eval_valide["prediction_generatif"] == df_eval_valide["prediction_bert"]

print(f"Erreurs communes aux deux modèles      : {((~df_eval_valide['generatif_correct']) & (~df_eval_valide['bert_correct'])).sum()}")
print(f"Erreurs uniquement du modèle génératif : {((~df_eval_valide['generatif_correct']) & (df_eval_valide['bert_correct'])).sum()}")
print(f"Erreurs uniquement de BERT             : {((df_eval_valide['generatif_correct']) & (~df_eval_valide['bert_correct'])).sum()}")
print(f"Désaccords entre les deux modèles      : {(~df_eval_valide['modeles_d_accord']).sum()} / {nb_test_valide}")

df_eval_valide[~df_eval_valide["modeles_d_accord"]][
    ["row_number", "verite", "prediction_generatif", "prediction_bert"]
]


## 11. Export du tableau d'évaluation complet

In [ ]:
FICHIER_SORTIE = "results/comparaison_generatif_bert.csv"
df_eval_valide.to_csv(FICHIER_SORTIE, index=False, encoding="utf-8")
print(f"Fichier '{FICHIER_SORTIE}' enregistré ({len(df_eval_valide)} lignes).")

print("\n=== Synthèse ===")
meilleur = NOM_GENERATIF if resultats_generatif["f1"] > resultats_bert["f1"] else NOM_BERT
print(f"Sur ces {nb_test_valide} lignes de test évaluables, le modèle avec le meilleur F1 macro est : {meilleur}.")
print("Attention : avec un échantillon de cette taille, un écart faible entre les deux modèles")
print("n'est pas forcément significatif statistiquement — à interpréter avec prudence.")
